In [ ]:
import kaggle_benchmarks as kbench
import json
import re
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

CANONICAL_NUMERIC_ANSWER = 81.7
ABS_TOL = 0.15  # tolerant enough for 3 s.f. style outputs

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def safe_get_attr(obj, attr_name, default=None):
    try:
        return getattr(obj, attr_name, default)
    except Exception:
        return default

def build_trace(
    *,
    task_id,
    llm,
    prompt,
    response,
    parsed,
    final_answer,
    normalized_answer,
    passed,
    failure_mode,
):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "normalized_answer": normalized_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt,
        "tokens_input": safe_get_attr(llm, "last_input_tokens"),
        "tokens_output": safe_get_attr(llm, "last_output_tokens"),
        "cost": safe_get_attr(llm, "last_cost"),
        "latency_ms": safe_get_attr(llm, "last_latency_ms"),
    }

def normalize_numeric_answer(text):
    if text is None:
        return ""

    s = str(text).strip().lower()

    replacements = {
        "°": "",
        "degrees": "",
        "degree": "",
        "\\degree": "",
        "\\circ": "",
        "deg": "",
        "$": "",
        ",": " ",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_first_number(text):
    s = normalize_numeric_answer(text)
    if not s:
        return None, s

    m = re.search(r"[-+]?\d+(?:\.\d+)?", s)
    if not m:
        return None, s

    try:
        return float(m.group(0)), s
    except Exception:
        return None, s

def code_verifier(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return False, "hallucination", normalized

    if abs(value - CANONICAL_NUMERIC_ANSWER) <= ABS_TOL:
        return True, None, normalized

    return False, None, normalized

def classify_failure_fp_0003(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return "hallucination"

    # Task-specific common wrong branches from uploaded results/analysis
    if abs(value - 52.7) <= 0.2:
        return "failure_to_recognize_key_aspects"

    if abs(value - 68.4) <= 0.2:
        return "misapplication_of_equation_or_model"

    # If it is numerically close-ish but not right, treat as calculation
    if abs(value - CANONICAL_NUMERIC_ANSWER) <= 10:
        return "calculation_error"

    # Very far-off but still numeric likely reflects wrong physical branch/model
    return "misapplication_of_equation_or_model"

# ----------------------------
# Frontier Physics Task 003
# ----------------------------
@kbench.task(
    name="FP-0003 Ellipsoidal Bowl With Rim Bead and Fence",
    description="Hard classical-mechanics equilibrium selection on a constrained ellipsoid with a rim bead, fence constraint, and weak symmetry-breaking tilt."
)
def fp_0003_ellipsoidal_bowl_rim_bead(llm) -> tuple[int, int]:
    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

A smooth rigid bowl is the lower half of the triaxial ellipsoid $\frac{x^2}{a^2}+\frac{y^2}{b^2}+\frac{z^2}{c^2}=1,\ z\le 0$ with gravity $\mathbf g=-g\,\hat{\mathbf z}$. A particle of mass $m_1$ moves frictionlessly on the inside surface. A massless inextensible string connects the particle at $Q=(x,y,z)$ to a frictionless bead at $P=(x_p,y_p,0)$ that can slide on the rim ellipse $\frac{x_p^2}{a^2}+\frac{y_p^2}{b^2}=1,\ z_p=0$. From $P$, the string runs to a fixed frictionless guide point $S$, then vertically down to a hanging mass $m_2$ at rest. Tension is uniform, $T=m_2 g$.

Fence constraint: the rim bead is constrained to remain on the arc with $y_p\ge 0$.

In addition to gravity, the particle experiences a uniform constant body force $\mathbf F_\epsilon=\epsilon\,m_1 g\,\hat{\mathbf y}$ with $\epsilon=0.0123$. There is arbitrarily weak linear drag $\mathbf F_d=-\gamma \mathbf v$ with $\gamma>0$, and the system is released from rest from any initial condition with $y(0)>0$ and $y_p(0)>0$. It settles as $t\to\infty$ to the unique stable equilibrium permitted by the fence.

Numerical data (SI):
$a=0.22,\ b=0.13,\ c=0.17\ \mathrm{m}$
$m_1=0.12,\ m_2=0.08\ \mathrm{kg}$
$S=(0.050,\ 0,\ 0.280)\ \mathrm{m}$
$\epsilon=0.0123$

At the limiting equilibrium $t\to\infty$, let $\theta$ be the angle between $\mathbf r_Q$ and the downward vertical axis $-\hat{\mathbf z}$, and let $\varphi=\operatorname{atan2}(y,x)$ in degrees.

Question: What is $\theta+\varphi$ in degrees, up to 3 significant figures?

Return JSON only in the following format:
{
  "final_answer": "<numeric value in degrees>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    normalized_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        code_result, code_failure, normalized_answer = code_verifier(final_answer)

        if code_result is True:
            passed_checks = 1
        else:
            failure_mode = code_failure or classify_failure_fp_0003(final_answer)

    trace = build_trace(
        task_id="fp_0003",
        llm=llm,
        prompt=prompt,
        response=response,
        parsed=parsed,
        final_answer=final_answer,
        normalized_answer=normalized_answer,
        passed=(passed_checks == 1),
        failure_mode=failure_mode,
    )
    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0003_ellipsoidal_bowl_rim_bead.run(kbench.llm)

In [ ]:
results = fp_0003_ellipsoidal_bowl_rim_bead.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df[trace_df["task_id"] == "fp_0003"]